# Planetary Rover GRPO Training (Colab)

This notebook runs the same GRPO pipeline as `train.py` for the OpenEnv hackathon submission.

Pipeline:
- Llama 3.2 1B + Unsloth 4-bit NF4 + LoRA
- TRL `GRPOTrainer`
- Two-tier rewards: format gatekeeper + environment reward from FastAPI physics server

## 1) Prepare Runtime



In [ ]:
import os
import pathlib
import subprocess

# Define the target directory
repo_path = pathlib.Path('/content/planetary-rover-navigation')

# Clone the repo if it doesn't exist
if not repo_path.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/Gurram-Bhaskar/planetary-rover-navigation.git", str(repo_path)], check=True)
else:
    print("Repository already exists.")

# Change into the directory
os.chdir(repo_path)
print('Working directory:', os.getcwd())

In [ ]:
# 1. Install Unsloth using its official Colab installer (handles most heavy lifting)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install core libraries WITHOUT forcing an old version of xformers.
# This allows pip to grab the pre-compiled version compatible with Colab's Torch.
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 3. Dynamically change strict "==" to flexible ">=" in your requirements.txt 
!sed -i 's/==/>=/g' requirements.txt

# 4. Install the remaining app dependencies quietly
!pip install -q -r requirements.txt

In [ ]:
import requests
import subprocess
import sys
import time

server_proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'
])

deadline = time.time() + 30
server_ready = False
while time.time() < deadline:
    try:
        res = requests.get('http://127.0.0.1:7860/tasks', timeout=2)
        if res.ok:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    raise RuntimeError('Server failed to start within 30 seconds.')

print('FastAPI environment server is live on :7860')

## 2) Run GRPO Training


In [ ]:
import os
import wandb
from google.colab import userdata

try:
    # Try to grab the W&B key from Colab Secrets (the key icon on the left)
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("Successfully logged into Weights & Biases!")
except userdata.SecretNotFoundError:
    print("No WANDB_API_KEY found in Colab secrets. Disabling W&B logging for this run.")
    os.environ['WANDB_MODE'] = 'disabled'


In [ ]:
import os
import train as rover_train

os.environ['ROVER_SERVER_URL'] = 'http://127.0.0.1:7860'
os.environ['ROVER_NUM_GENERATIONS'] = '8'

rover_train.SERVER_URL = os.environ['ROVER_SERVER_URL']
rover_train.NUM_GENERATIONS = int(os.environ['ROVER_NUM_GENERATIONS'])

rover_train.main()

In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('Stopped FastAPI environment server.')
else:
    print('No running server process found.')